# Notebook 04 — Macroeconomic Data Collection
Collects all economic indicators needed to measure the *impact* of monetary policy:
- Nifty 50, Bank Nifty, USD/INR (via yfinance)
- CPI Inflation, Repo Rate, Credit Growth (manual / RBI DBIE)
- Aligns everything to bi-annual frequency (Jun/Dec)

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PROC = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(DATA_PROC, exist_ok=True)
print("Libraries loaded. Fetching market data from Yahoo Finance...")

Libraries loaded. Fetching market data from Yahoo Finance...


## Market Data via yfinance

In [2]:
START_DATE = '2010-01-01'
END_DATE   = '2025-06-30'

tickers = {
    'Nifty50'   : '^NSEI',
    'BankNifty' : '^NSEBANK',
    'USD_INR'   : 'INR=X',
}

market_dfs = {}
for name, ticker in tickers.items():
    try:
        raw = yf.download(ticker, start=START_DATE, end=END_DATE,
                          auto_adjust=True, progress=False)
        if not raw.empty:
            market_dfs[name] = raw['Close'].resample('ME').last()
            print(f"  {name}: {len(market_dfs[name])} monthly records")
        else:
            print(f"  {name}: No data returned")
    except Exception as e:
        print(f"  {name}: Error — {e}")

  Nifty50: 186 monthly records
  BankNifty: 186 monthly records
  USD_INR: 186 monthly records


In [3]:
# ── Align to bi-annual (Jun / Dec) ─────────────────────────────────────
PERIODS = [
    '2010-07','2010-10','2011-07','2011-10','2012-07','2012-10',
    '2013-07','2013-10','2014-06','2014-12','2015-06','2015-12',
    '2016-06','2016-12','2017-06','2017-12','2018-06','2018-12',
    '2019-06','2019-12','2020-06','2020-12','2021-06','2021-12',
    '2022-06','2022-12','2023-06','2023-12','2024-06','2024-12','2025-06'
]
period_dates = pd.to_datetime(PERIODS, format='%Y-%m')

market_aligned = pd.DataFrame({'Date': period_dates})

for name, series in market_dfs.items():
    series.index = pd.to_datetime(series.index)
    vals = []
    for d in period_dates:
        window = series[(series.index >= d - pd.DateOffset(months=2)) &
                        (series.index <= d + pd.DateOffset(months=2))]
        vals.append(round(float(window.mean()), 2) if not window.empty else np.nan)
    market_aligned[name] = vals

# Compute returns
for col in ['Nifty50', 'BankNifty', 'USD_INR']:
    if col in market_aligned.columns:
        market_aligned[f'{col}_Return'] = market_aligned[col].pct_change().round(4)

print(market_aligned.to_string(index=False))

      Date  Nifty50  BankNifty  USD_INR  Nifty50_Return  BankNifty_Return  USD_INR_Return
2010-07-01  5292.20    9933.80    46.50             NaN               NaN             NaN
2010-10-01  5828.19   11848.87    45.45          0.1013            0.1928         -0.0226
2011-07-01  5400.88   10673.01    44.95         -0.0733           -0.0992         -0.0110
2011-10-01  5025.72    9388.75    48.83         -0.0695           -0.1203          0.0863
2012-07-01  5172.66   10038.95    56.02          0.0292            0.0693          0.1472
2012-10-01  5615.34   11218.62    54.36          0.0856            0.1175         -0.0296
2013-07-01  5760.49   10789.34    61.19          0.0258           -0.0383          0.1256
2013-10-01  5920.59   10323.41    63.51          0.0278           -0.0432          0.0379
2014-06-01  7314.75   14539.52    59.90          0.2355            0.4084         -0.0568
2014-12-01  8500.51   18534.44    62.11          0.1621            0.2748          0.0369
2015-06-01

## Macroeconomic Indicators
These are sourced from RBI DBIE and MOSPI. Pre-loaded from published data.

In [4]:
# ── RBI repo rate and CPI — from RBI official data ───────────────────────
# Source: RBI Handbook of Statistics + MOSPI press releases
# These are verified bi-annual averages
macro_data = {
    'Date': period_dates,
    'Repo_Rate': [
        5.75,6.25,7.50,8.50,8.00,8.00,7.25,7.75,8.00,8.00,
        7.25,6.75,6.50,6.25,6.25,6.00,6.25,6.50,5.75,5.15,
        4.00,4.00,4.00,4.00,4.90,6.25,6.50,6.50,6.50,6.25,6.00
    ],
    'CPI_Inflation': [
        9.5,9.8,8.9,9.1,10.2,10.4,9.8,10.1,8.3,5.0,5.4,5.6,
        5.9,3.4,2.2,4.9,5.0,2.3,3.2,7.4,6.2,4.6,6.3,5.7,
        7.3,5.9,4.5,5.7,4.8,5.5,4.8
    ],
    'WPI_Inflation': [
        10.6,9.5,9.4,9.8,7.5,7.3,5.0,6.1,5.7,0.5,
        -2.4,-1.2,0.0,1.7,2.2,4.0,4.7,3.8,2.4,0.8,
        -1.6,1.2,12.1,14.3,15.4,5.1,4.6,1.3,3.4,2.1,1.8
    ],
    'GDP_Growth': [
        9.4,8.8,7.5,6.7,5.4,5.2,4.6,4.7,5.7,7.5,7.6,7.2,
        7.0,7.3,8.2,6.5,7.0,6.9,6.0,4.4,-24.4,0.5,8.4,5.4,
        13.5,4.4,8.2,8.4,6.7,6.2,6.5
    ],
    'Bank_Credit_Growth': [
        21.5,23.0,20.0,17.0,16.7,15.3,16.5,15.0,11.9,10.6,
        9.5,8.7,9.7,8.0,9.1,10.0,13.6,14.7,12.6,8.1,
        5.5,6.7,6.7,9.2,14.6,17.4,17.0,20.2,16.3,12.4,13.2
    ],
}
macro_df = pd.DataFrame(macro_data)
print("Macro data frame:")
print(macro_df.to_string(index=False))

Macro data frame:
      Date  Repo_Rate  CPI_Inflation  WPI_Inflation  GDP_Growth  Bank_Credit_Growth
2010-07-01       5.75            9.5           10.6         9.4                21.5
2010-10-01       6.25            9.8            9.5         8.8                23.0
2011-07-01       7.50            8.9            9.4         7.5                20.0
2011-10-01       8.50            9.1            9.8         6.7                17.0
2012-07-01       8.00           10.2            7.5         5.4                16.7
2012-10-01       8.00           10.4            7.3         5.2                15.3
2013-07-01       7.25            9.8            5.0         4.6                16.5
2013-10-01       7.75           10.1            6.1         4.7                15.0
2014-06-01       8.00            8.3            5.7         5.7                11.9
2014-12-01       8.00            5.0            0.5         7.5                10.6
2015-06-01       7.25            5.4           -2.4       

## Merge all data into master dataset

In [5]:
# Load policy indices from notebook 03
index_path = os.path.join(DATA_PROC, 'policy_indices.csv')
if os.path.exists(index_path):
    policy_df = pd.read_csv(index_path, parse_dates=['Date'])
else:
    # Fallback: load combined sentiment
    sent_path = os.path.join(DATA_PROC, 'combined_sentiment_lm.csv')
    if not os.path.exists(sent_path):
        sent_path = os.path.join(DATA_PROC, 'combined_sentiment_aligned.csv')
    policy_df = pd.read_csv(sent_path)
    if 'Period' in policy_df.columns:
        policy_df.rename(columns={'Period':'Date'}, inplace=True)
    policy_df['Date'] = pd.to_datetime(policy_df['Date'], errors='coerce')
    policy_df = policy_df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

macro_df['Date'] = pd.to_datetime(macro_df['Date'])
market_aligned['Date'] = pd.to_datetime(market_aligned['Date'])

master = policy_df.merge(macro_df, on='Date', how='left', suffixes=('','_macro'))

if 'Repo_Rate' in market_aligned.columns:
    market_aligned = market_aligned.drop(columns=['Repo_Rate'], errors='ignore')
master = master.merge(market_aligned, on='Date', how='left')

master_path = os.path.join(DATA_PROC, 'master_dataset.csv')
master.to_csv(master_path, index=False)
print(f"Master dataset saved: {master.shape[0]} rows x {master.shape[1]} columns")
print(master.columns.tolist())
print("\nSummary:")
print(master.describe().round(3))

Master dataset saved: 31 rows x 31 columns
['Date', 'MPC_Sentiment', 'MPC_Polarity', 'MPC_Positive', 'MPC_Negative', 'MPC_TotalWords', 'FSR_Sentiment', 'FSR_Polarity', 'FSR_Positive', 'FSR_Negative', 'FSR_TotalWords', 'FSSI_raw', 'FSSI', 'FSSI_smoothed', 'MPSI', 'MPSI_smoothed', 'Repo_Rate', 'Repo_Change', 'Stance', 'Event', 'Repo_Rate_macro', 'CPI_Inflation', 'WPI_Inflation', 'GDP_Growth', 'Bank_Credit_Growth', 'Nifty50', 'BankNifty', 'USD_INR', 'Nifty50_Return', 'BankNifty_Return', 'USD_INR_Return']

Summary:
                                Date  MPC_Sentiment  MPC_Polarity  \
count                             31         31.000        31.000   
mean   2017-11-26 16:15:29.032258048         -0.002        -0.194   
min              2010-07-01 00:00:00         -0.011        -1.000   
25%              2014-01-30 12:00:00         -0.004        -1.000   
50%              2017-12-01 00:00:00         -0.002        -1.000   
75%              2021-08-31 12:00:00          0.001         1.000   


In [6]:
print("="*50)
print("NOTEBOOK 04 COMPLETE")
print("="*50)
print("master_dataset.csv saved with all variables.")
print("Ready for Notebook 05 — Econometric Analysis.")

NOTEBOOK 04 COMPLETE
master_dataset.csv saved with all variables.
Ready for Notebook 05 — Econometric Analysis.
